# scene01 proof of concept: changes, uncertainty and robot heat maps (Colab, one T4)

Research paths P1 (shared reconstruction) and P2 (change detection), with the P3-P6 outputs layered on top.
Branch `exp/change/poc-lowcompute`. Runs until the session ends; every finished step is pushed to GitHub and
skipped by later sessions (any Google account), so just **Run all** again to continue.

**Method (low compute, no training):** VGGT-1B reconstructs frames from both recordings jointly in one shared 3D
frame (~1-2 min per pair on a T4). Every point is projected into the other recording's cameras: it is *gone* if
they see through it, *occluded* if something is in front, *appearance-changed* if the surface agrees but DINOv2
features differ. Thresholds come from comparing views of the **same** recording (nothing changed there), so no
labels are used. Changed points are clustered into objects, re-identified across recordings (moved = removed +
added that look alike) and labelled with CLIP.

**Per pair it writes** (`results/change/<RUN_ID>/<step>/`):
- `heatmaps.png`: surface uncertainty (cm), surface shift / deformation (cm), height, robot reachability,
  gripper approachability, interactability
- `base_placement.png`: where on the floor a robot could stand and how much of the scene (and of the changes) it reaches
- `change_map.png`, `overlays.png`, `crops.png`, `calibration.png`
- `viewer.html`: interactive 3D with toggleable change and heat-map layers (download and open in a browser)
- `scene_report.md`: changes with metric sizes and uncertainty, robot section, simulator randomisation ranges
- `cloud.ply`, `fields.ply`, `metrics.json`, `changes.json`, `report.md`
- `summary_*` steps: one table and figure across all pairs (false changes on static/lighting pairs vs real changes)

**Setup:** Runtime -> Change runtime type -> **T4 GPU**. Secrets panel (key icon): `GITHUB_TOKEN` (fine-grained,
Contents read/write on UncertainScene) with notebook access on. Never paste a token into a cell.

Metric scale assumes the phone was held at 1.35 m (range 1.1-1.6 m, carried as uncertainty).

In [ ]:
# ---- Config ---------------------------------------------------------------------------
BRANCH = "exp/change/poc-lowcompute"
RUN_ID = "2026-10-10_poc-scene01"   # keep the same id to resume; change it only for a fresh run
PATH_NAME = "change"
EXPECTED_COMMIT = ""                # optional: short sha you expect; setup stops if HEAD differs
RETRY_FAILED = False                # True re-runs steps that failed with an error (after a code fix)
ONLY_PHASES = None                  # e.g. ["p1", "p2"] to restrict; None = the whole queue
SHOW_FIGURES = True                 # show heat maps inline as steps finish
REPO = "FinPThomas/UncertainScene"

In [ ]:
# ---- Setup: token, clone to local disk, commit check, push check ------------------------
import base64, os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

token = secret("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN in the Secrets panel and turn on notebook access"
os.environ["GITHUB_TOKEN"] = token
label = secret("UNCERTAINSCENE_ACCOUNT")
if label:
    os.environ["UNCERTAINSCENE_ACCOUNT"] = label

ROOT = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
REPO_DIR = ROOT / "UncertainScene"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"]  # never stored in .git/config

def git(*args, cwd=REPO_DIR):
    p = subprocess.run(["git", *AUTH, *args], cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr.replace(token, "***").replace(auth, "***"))
    return p.stdout.strip()

if not (REPO_DIR / ".git").exists():
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(REPO_DIR), cwd=ROOT)
else:
    git("fetch", "origin", BRANCH)
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
head = git("rev-parse", "--short", "HEAD")
print(f"On {BRANCH} at {head}:", git("log", "-1", "--format=%s"))
if EXPECTED_COMMIT and not head.startswith(EXPECTED_COMMIT[:7]):
    raise SystemExit(f"Stale code: HEAD is {head}, expected {EXPECTED_COMMIT}")

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-e", f"{REPO_DIR}[poc]"], check=True)
subprocess.run(pip + ["--no-deps", "git+https://github.com/facebookresearch/vggt.git"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from uncertainscene.infra.publish import Publisher
pub = Publisher(REPO_DIR, BRANCH, token)
pub.ensure_identity()
pub.check_push_access()
print("Push access OK")

In [ ]:
# ---- GPU check and run state ------------------------------------------------------------
import logging, torch
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from uncertainscene.infra import Run
from uncertainscene.infra.guard import ResourceLimit, read_usage
from uncertainscene.infra.store import Store, repo_slug

run = Run.from_env(run_id=RUN_ID, path=PATH_NAME, branch=BRANCH)
print("Session label:", run.account)
print("Machine:", read_usage(run.budget.work_dir).line())
print("Run state:\n" + run.state.summary())

CACHE = Path(run.budget.work_dir) / "cache"      # reconstructions reused across steps (outside step scratch)
RUN_DIR = REPO_DIR / "results" / PATH_NAME / RUN_ID
try:
    store = Store(repo_slug(REPO_DIR))            # dense clouds as release assets (best effort)
except Exception as e:
    store = None
    print("No release store:", e)

In [ ]:
# ---- The queue: runs until the session ends ---------------------------------------------
# Order: p1 proof of concept with every heat map -> p2 dense versions -> p3 robot scenarios ->
# p4 two sanity ablations -> p5 every other take combination -> p6 fewer frames -> p7 repeats.
# Each step: skipped if done (by any account) or claimed by a live session elsewhere; not started
# if the RAM, disk or session time it needs is not available.
from IPython.display import Image, Markdown, display
from uncertainscene.eval import summarize
from uncertainscene.experiments.poc import analyse, queue, trim_cache

def summary_step(out, work):
    md = summarize.write(RUN_DIR, out)
    return f"summary of {len(summarize.load(RUN_DIR))} steps"

counts = {}
try:
    for item in queue():
        name = item if isinstance(item, str) else item.name
        if ONLY_PHASES and name.split("_")[0] not in ONLY_PHASES and not name.startswith("summary_" + "_".join(ONLY_PHASES)):
            continue
        if isinstance(item, str):
            status = run.step(item, summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2), retry_failed=RETRY_FAILED)
        else:
            status = run.step(item.name, lambda out, work, spec=item: analyse(spec, out, CACHE, store),
                              needs=item.needs, retry_failed=RETRY_FAILED)
            trim_cache(CACHE, max_gb=6)
            torch.cuda.empty_cache()
            hm = run.out_dir(item.name) / "heatmaps.png"
            if SHOW_FIGURES and status == "done" and item.outputs == "full" and hm.exists():
                display(Markdown(f"### {item.name}\n{run.state.step(item.name).note}"), Image(str(hm), width=1100))
        counts[status] = counts.get(status, 0) + 1
        print(f"{name}: {status}  {run.state.step(name).note or ''}")
except ResourceLimit as e:
    print("Stopping cleanly (results so far are pushed):", e)
print(counts)

In [ ]:
# ---- Finish: refresh the overall summary and show the headline ---------------------------
try:
    run.step(f"summary_session_{run.account}", summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2))
except ResourceLimit as e:
    print("No time left for the final summary:", e)
run.close()
print(run.state.summary())
s = run.out_dir(f"summary_session_{run.account}") / "summary.md"
if s.exists():
    display(Markdown(s.read_text()[:6000]))